In [1]:
%cd ..

/home/v1adych/projects/kolobok


In [66]:
from pathlib import Path
from typing import Tuple, Optional

import cv2
import numpy as np
import matplotlib.pyplot as plt

import pandas as pd

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.io import read_image
from torchvision import transforms

from inference import get_model

In [67]:
seg_model = get_model("tire-thread-detection/3", api_key="BRdDttL8wwHFrA27Xv07")

/home/v1adych/micromamba/envs/tyro/lib/python3.11/site-packages/onnxruntime/capi/onnxruntime_inference_collection.py:118: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
/home/v1adych/micromamba/envs/tyro/lib/python3.11/site-packages/onnxruntime/capi/onnxruntime_inference_collection.py:118: UserWarning: Specified provider 'OpenVINOExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
/home/v1adych/micromamba/envs/tyro/lib/python3.11/site-packages/onnxruntime/capi/onnxruntime_inference_collection.py:118: UserWarning: Specified provider 'CoreMLExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


In [68]:
def get_thread_crop(img: np.ndarray) -> np.ndarray:
    preds = seg_model.infer(img)[0]
    min_x, max_x = img.shape[1], 0
    min_y, max_y = img.shape[0], 0
    padding = (int(0.1 * img.shape[0]), int(0.1 * img.shape[1]))
    if len(preds.predictions) > 1:
        print(
            f"Warning: {len(preds.predictions)} predictions found, using the first one"
        )
    if len(preds.predictions) == 0:
        raise ValueError("No predictions found")

    for pred in preds.predictions:
        if pred.class_name != "tire":
            continue
        min_x = min(min_x, int(pred.x - pred.width / 2))
        max_x = max(max_x, int(pred.x + pred.width / 2))
        min_y = min(min_y, int(pred.y - pred.height / 2))
        max_y = max(max_y, int(pred.y + pred.height / 2))
        break

    return img[
        max(min_y - padding[0], 0) : min(max_y + padding[0], img.shape[0]),
        max(min_x - padding[1], 0) : min(max_x + padding[1], img.shape[1]),
    ]

In [69]:
data_root = Path("data/dataset")
processed_data_root = Path("data/processed_dataset")
processed_data_root.mkdir(exist_ok=True, parents=True)

In [70]:
# data = []

# for dir in data_root.iterdir():
#     label = float(dir.name[:-2].replace(",", "."))
#     for image_path in dir.iterdir():
#         img = cv2.imread(str(image_path))
        
#         try:
#             img = get_thread_crop(img)
#         except ValueError:
#             print(f"Error processing {image_path}")
#             continue

#         cv2.imwrite(str(processed_data_root / f"{len(data)}.png"), img)
#         data.append([f"{len(data)}.png", label])

Error processing data/dataset/1,4мм/20240626_172732.jpg
Error processing data/dataset/3,2мм/2022-07-15 14-32-29.JPG
Error processing data/dataset/3,2мм/20240928_211046.jpg
Error processing data/dataset/3,2мм/20240702_124724.jpg
Error processing data/dataset/3,2мм/20240809_153007.jpg
Error processing data/dataset/3,2мм/20230310_122427.jpg
Error processing data/dataset/3,2мм/20231108_105613.jpg
Error processing data/dataset/3,2мм/20230111_110213.jpg
Error processing data/dataset/3,2мм/20231108_105415.jpg
Error processing data/dataset/3,2мм/20240626_130303.jpg
Error processing data/dataset/3,2мм/2022-07-25 15-05-23.JPG
Error processing data/dataset/3,2мм/20231003_120329.jpg
Error processing data/dataset/3,2мм/20240713_154655.jpg
Error processing data/dataset/2,7мм/20241213_012608.jpg
Error processing data/dataset/2,7мм/20230929_155934.jpg
Error processing data/dataset/2,7мм/20230914_130417.jpg
Error processing data/dataset/2,7мм/20230120_164157.jpg
Error processing data/dataset/2,7мм/2023

In [ ]:
# df = pd.DataFrame(data, columns=["image", "label"])
# df.to_csv(processed_data_root / "labels.csv", index=False)

In [72]:
df = pd.read_csv(processed_data_root / "labels.csv")

In [73]:
class ThreadDataset(Dataset):
    def __init__(self, data: pd.DataFrame, data_root_dir: str, transform: Optional[nn.Module] = None):
        self.data = data
        self.data_root_dir = Path(data_root_dir)
        self.image_paths = []
        self.labels = []
        for _, row in self.data.iterrows():
            image_path = self.data_root_dir / row["image"]
            if not image_path.exists():
                print(f"Warning: {image_path} does not exist")
                continue
            self.image_paths.append(self.data_root_dir / row["image"])
            self.labels.append(row["label"])

        self.transform = transform
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, float]:
        image_path = self.image_paths[idx]
        label = self.labels[idx]
        
        image = read_image(str(image_path)) / 255
        if self.transform is not None:
            image = self.transform(image)
        return image, label

In [74]:
dataset = ThreadDataset(df, processed_data_root)

In [184]:
class CropPadding(nn.Module):
    def __init__(self, num_pixels: int):
        super().__init__()
        self.num_pixels = num_pixels

    def forward(self, image: torch.Tensor) -> torch.Tensor:
        *_, h, w = image.shape
        return image[
            ...,
            self.num_pixels : h - self.num_pixels,
            self.num_pixels : w - self.num_pixels,
        ]


augmentation_transform = transforms.Compose(
    [
        transforms.Pad(100, padding_mode="reflect"),
        transforms.ToPILImage(),
        transforms.RandAugment(
            num_ops=4, interpolation=transforms.InterpolationMode.BICUBIC
        ),
        transforms.ToTensor(),
        CropPadding(100),
    ]
)

train_dataset, val_dataset = torch.utils.data.random_split(
    dataset,
    [0.8, 0.2],
    generator=torch.Generator().manual_seed(42),
)
train_dataset.dataset.transform = augmentation_transform

In [185]:
train_loader = DataLoader(
    train_dataset,
    shuffle=True,
)
val_loader = DataLoader(
    val_dataset,
    shuffle=False,
)

In [186]:
model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Sequential(nn.Linear(2048, 512), nn.ReLU(), nn.Linear(512, 1))

In [188]:
def train_fn(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader, num_epochs: int = 10, grad_accumulation_steps: int = 1):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for i, (images, labels) in enumerate(train_loader):
            images = images.to(device)
            labels = labels.to(device).unsqueeze(1)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels) / grad_accumulation_steps
            loss.backward()

            if (i + 1) % grad_accumulation_steps == 0:
                optimizer.step()

            running_loss += loss.item()

        print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {running_loss / len(train_loader):.4f}")

        model.eval()
        mae = []
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device).unsqueeze(1)

                outputs = model(images)
                mae.append(torch.abs(outputs - labels).mean().item())
        print(f"Validation MAE: {np.mean(mae):.4f}")

In [189]:
train_fn(
    model,
    train_loader,
    val_loader,
    num_epochs=10,
    grad_accumulation_steps=4,
)

OutOfMemoryError: CUDA out of memory. Tried to allocate 120.00 MiB. GPU 0 has a total capacity of 3.81 GiB of which 70.75 MiB is free. Including non-PyTorch memory, this process has 3.73 GiB memory in use. Of the allocated memory 3.53 GiB is allocated by PyTorch, and 121.47 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)